In [6]:


import csv
from decimal import Decimal, ROUND_HALF_UP
from pathlib import Path
from itertools import zip_longest

# ==================================================
# 1. File paths
# ==================================================

input_file = Path("45stocks.csv")
output_file = Path("45stocks_v1.csv")

# ==================================================
# 2. Columns to adjust
# ==================================================

price_columns = [
    "DlyOpen",
    "DlyHigh",
    "DlyLow",
    "DlyClose",
    "DlyBid",
    "DlyAsk"
]

volume_column = "DlyVol"
factor_column = "DlyCumFacShr"

# Keep 8 decimal places for adjusted prices
precision = Decimal("0.00000001")

total_rows = 0
adjusted_rows = 0
adjusted_stocks = set()
all_stocks = set()

# ==================================================
# 3. Split adjustment
# ==================================================

with input_file.open(
    "r", encoding="utf-8-sig", newline=""
) as fin, output_file.open(
    "w", encoding="utf-8", newline=""
) as fout:

    reader = csv.DictReader(fin)
    columns = reader.fieldnames

    required = set(
        price_columns +
        [volume_column, factor_column,
         "PERMNO", "Ticker", "DlyCalDt"]
    )

    assert columns is not None
    assert len(columns) == 16
    assert required.issubset(columns)

    writer = csv.DictWriter(
        fout,
        fieldnames=columns,
        lineterminator="\n"
    )
    writer.writeheader()

    for row in reader:

        total_rows += 1
        ticker = row["Ticker"]
        all_stocks.add(ticker)

        factor = Decimal(row[factor_column])

        if factor <= 0:
            raise ValueError(
                f"Invalid factor: {ticker}, "
                f"{row['DlyCalDt']}"
            )

        # No split adjustment needed when factor = 1
        if factor != 1:

            adjusted_rows += 1
            adjusted_stocks.add(ticker)

            # Adjust OHLC and Bid/Ask
            for col in price_columns:

                adjusted_price = (
                    Decimal(row[col]) / factor
                ).quantize(
                    precision,
                    rounding=ROUND_HALF_UP
                )

                row[col] = format(adjusted_price, "f")

            # Adjust volume
            adjusted_volume = (
                Decimal(row[volume_column]) * factor
            )

            if adjusted_volume != adjusted_volume.to_integral_value():
                raise ValueError(
                    f"Non-integer adjusted volume: "
                    f"{ticker}, {row['DlyCalDt']}"
                )

            row[volume_column] = format(
                adjusted_volume, ".0f"
            )

        writer.writerow(row)

# ==================================================
# 4. Validate unchanged columns and row order
# ==================================================

modified_columns = set(price_columns + [volume_column])

with input_file.open(
    "r", encoding="utf-8-sig", newline=""
) as fin, output_file.open(
    "r", encoding="utf-8", newline=""
) as fout:

    original = csv.DictReader(fin)
    adjusted = csv.DictReader(fout)

    assert original.fieldnames == adjusted.fieldnames

    unchanged_columns = [
        c for c in original.fieldnames
        if c not in modified_columns
    ]

    for old, new in zip_longest(original, adjusted):

        assert old is not None and new is not None

        for col in unchanged_columns:
            assert old[col] == new[col], (
                f"Unexpected change in {col}"
            )

        if Decimal(old[factor_column]) == 1:
            assert old == new

# ==================================================
# 5. Final verification
# ==================================================

assert total_rows == 124470
assert len(all_stocks) == 45

print("Split adjustment completed.")
print(f"Total stocks: {len(all_stocks)}")
print(f"Total rows: {total_rows:,}")
print(f"Rows requiring adjustment: {adjusted_rows:,}")
print(f"Stocks requiring adjustment: {len(adjusted_stocks)}")
print(f"Adjusted stocks: {sorted(adjusted_stocks)}")
print(f"Output: {output_file}")
print("All other columns and row order preserved.")


Split adjustment completed.
Total stocks: 45
Total rows: 124,470
Rows requiring adjustment: 15,339
Stocks requiring adjustment: 11
Adjusted stocks: ['AAPL', 'AMZN', 'AVGO', 'NEE', 'NKE', 'NVDA', 'SBUX', 'SRE', 'TJX', 'V', 'WMT']
Output: 45stocks_v1.csv
All other columns and row order preserved.


In [7]:

import csv
from pathlib import Path

# 1. File paths
input_file = Path("45stocks_v1.csv")
output_file = Path("45stocks_v2.csv")

# 2. Columns to remove
remove_columns = {"DlyBid", "DlyAsk"}

# 3. Remove columns
with input_file.open("r", encoding="utf-8-sig", newline="") as fin, \
     output_file.open("w", encoding="utf-8", newline="") as fout:

    reader = csv.DictReader(fin)
    original_columns = reader.fieldnames

    assert original_columns is not None
    assert len(original_columns) == 16
    assert remove_columns.issubset(original_columns)

    remaining_columns = [
        col for col in original_columns
        if col not in remove_columns
    ]

    writer = csv.DictWriter(
        fout,
        fieldnames=remaining_columns,
        lineterminator="\n"
    )
    writer.writeheader()

    total_rows = 0

    for row in reader:
        writer.writerow({
            col: row[col] for col in remaining_columns
        })
        total_rows += 1

# 4. Validation
assert total_rows == 124470
assert len(remaining_columns) == 14
assert not remove_columns.intersection(remaining_columns)

print("Column removal completed.")
print(f"Total rows: {total_rows:,}")
print(f"Original columns: {len(original_columns)}")
print(f"Remaining columns: {len(remaining_columns)}")
print(f"Removed columns: {sorted(remove_columns)}")
print(f"Output: {output_file}")


Column removal completed.
Total rows: 124,470
Original columns: 16
Remaining columns: 14
Removed columns: ['DlyAsk', 'DlyBid']
Output: 45stocks_v2.csv


In [9]:
import pandas as pd

df = pd.read_csv("45stocks_v2.csv")

print(df.columns.tolist())

['PERMNO', 'Ticker', 'DlyCalDt', 'DlyOpen', 'DlyHigh', 'DlyLow', 'DlyClose', 'DlyVol', 'DlyCumFacPr', 'DlyCumFacShr', 'DlyRet', 'DlyRetx', 'DlyCap', 'sprtrn']


In [10]:
import csv
from pathlib import Path

# 1. File paths
input_file = Path("45stocks_v2.csv")
output_file = Path("45stocks_v3.csv")

# 2. Columns to remove
remove_columns = {
    "DlyCumFacPr",
    "DlyCumFacShr",
    "DlyRet",
    "DlyRetx"
}

# 3. Remove columns
with input_file.open("r", encoding="utf-8-sig", newline="") as fin, \
     output_file.open("w", encoding="utf-8", newline="") as fout:

    reader = csv.DictReader(fin)
    original_columns = reader.fieldnames

    assert original_columns is not None
    assert len(original_columns) == 14
    assert remove_columns.issubset(original_columns)

    remaining_columns = [
        col for col in original_columns
        if col not in remove_columns
    ]

    writer = csv.DictWriter(
        fout,
        fieldnames=remaining_columns,
        lineterminator="\n"
    )
    writer.writeheader()

    total_rows = 0

    for row in reader:
        writer.writerow({
            col: row[col] for col in remaining_columns
        })
        total_rows += 1

# 4. Validation
assert total_rows == 124470
assert len(remaining_columns) == 10
assert not remove_columns.intersection(remaining_columns)

print("Column removal completed.")
print(f"Total rows: {total_rows:,}")
print(f"Original columns: {len(original_columns)}")
print(f"Remaining columns: {len(remaining_columns)}")
print(f"Removed columns: {sorted(remove_columns)}")
print(f"Output: {output_file}")
print(f"Final columns: {remaining_columns}")

Column removal completed.
Total rows: 124,470
Original columns: 14
Remaining columns: 10
Removed columns: ['DlyCumFacPr', 'DlyCumFacShr', 'DlyRet', 'DlyRetx']
Output: 45stocks_v3.csv
Final columns: ['PERMNO', 'Ticker', 'DlyCalDt', 'DlyOpen', 'DlyHigh', 'DlyLow', 'DlyClose', 'DlyVol', 'DlyCap', 'sprtrn']
